## Section 1 — Setup: Clone repo, install deps, GPU validation


In [ ]:
# Trở về thư mục gốc của Colab
%cd /content

# Xóa hoàn toàn folder project cũ (bao gồm cả data và models bên trong)
!rm -rf Sentiment-Analysis-Service

!git clone https://github.com/RocketUp-Team/Sentiment-Analysis-Service.git || true
%cd Sentiment-Analysis-Service
!git checkout feature/ai-core
!git pull
!pip uninstall -y torchvision torchaudio
!pip install -r requirements.txt
!pip uninstall -y diffusers
!pip uninstall -y onnxruntime onnxruntime-gpu
!pip install onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/ --force-reinstall
!nvidia-smi


## Section 2 — Credentials: Read Colab Secrets, pre-flight checks


In [ ]:
import os
from google.colab import userdata

os.environ['MLFLOW_TRACKING_URI'] = userdata.get('MLFLOW_TRACKING_URI')
os.environ['DAGSHUB_USER'] = userdata.get('DAGSHUB_USER')
os.environ['DAGSHUB_TOKEN'] = userdata.get('DAGSHUB_TOKEN')
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
os.environ['MODEL_VERSION'] = userdata.get('MODEL_VERSION')

os.environ['MLFLOW_TRACKING_USERNAME'] = os.environ['DAGSHUB_USER']
os.environ['MLFLOW_TRACKING_PASSWORD'] = os.environ['DAGSHUB_TOKEN']

print("Secrets loaded.")

## Section 3 — Data Download


In [ ]:
!python -m src.data.downloader --task sarcasm
!python -m src.data.downloader --task sentiment


## Section 4 — Training


In [ ]:
import mlflow
from pathlib import Path
from src.scripts.run_finetuning import train
import os

model_version = os.environ.get("MODEL_VERSION", "v1.0.0")

def _adapter_exists(task_name):
    return Path(f"models/adapters/{task_name}").exists()

mlflow.set_experiment("colab_pipeline_run")

run_tags = {
    "model_version": model_version,
    "environment": "colab",
    "dataset_version": "v1"
}

with mlflow.start_run(run_name=f"full_pipeline_{model_version}") as parent_run:
    mlflow.set_tags(run_tags)

    for task in ["sarcasm", "sentiment"]:
        if not _adapter_exists(task):
            with mlflow.start_run(run_name=f"train_{task}_{model_version}", nested=True):
                mlflow.set_tags(run_tags)
                train(task)
        else:
            print(f"Skipping {task} training, adapter exists.")

## Section 5 — Evaluation


In [ ]:
from src.scripts.evaluate_finetuned import evaluate

with mlflow.start_run(run_id=parent_run.info.run_id):
    metrics_sarcasm = evaluate("sarcasm")
    mlflow.log_metric("sarcasm_overall_f1", metrics_sarcasm["overall_f1"])

    metrics_sentiment = evaluate("sentiment")
    mlflow.log_metric("sentiment_overall_f1", metrics_sentiment["overall_f1"])


## Section 6 — ONNX Export


In [ ]:
!python -m src.scripts.export_onnx --adapter-name sentiment
!python -m src.scripts.export_onnx --adapter-name sarcasm
!python -m src.scripts.benchmark_onnx


## Section 7 — Visualization


In [ ]:
!python -m src.scripts.generate_shap_plots --output-dir reports/shap_plots

import mlflow

try:
    with mlflow.start_run(run_id=parent_run.info.run_id):
        mlflow.log_artifacts("reports", artifact_path="final_reports")
        print("✅ Visualizations generated and successfully logged to MLflow Artifacts!")
except NameError:
    print("❌ Lỗi: Không tìm thấy parent_run. Hãy đảm bảo bạn đã chạy Section 4 trước.")

## Section 8 — DVC Push + Git Push


In [ ]:
!dvc remote modify origin --local auth basic
!dvc remote modify origin --local user $DAGSHUB_USER
!dvc remote modify origin --local password $DAGSHUB_TOKEN

!dvc commit -f \
    download_sarcasm \
    download_sentiment \
    finetune_sarcasm \
    finetune_sentiment \
    export_onnx_sarcasm \
    export_onnx_sentiment \
    benchmark_onnx

!dvc push

!git config --global user.email "trungdq.ts@gmail.com"
!git config --global user.name "TrungDQ"

!git remote set-url origin https://$GITHUB_TOKEN@github.com/RocketUp-Team/Sentiment-Analysis-Service.git

!git add dvc.lock reports/ models/onnx/ notebooks/colab_full_pipeline.ipynb || true
!git commit -m "chore: update dvc.lock for model-$MODEL_VERSION"
!git tag model-$MODEL_VERSION
!git push origin feature/ai-core
!git push origin model-$MODEL_VERSION
